# Temporial Graph RAG — bridge notebook (this codebase)

This notebook is a **companion** to the OpenAI cookbook **[`temporal_agents.ipynb`](../input_docs/temporal_agents.ipynb)** in `docs/input_docs/`. That file stays a **frozen reference**; this notebook shows how the **same ideas** map to **this repository** (FastAPI, `LLMClient`, Neo4j, ontologies, posted chunks).

**Run with:** from the repository root, install dev extras and start JupyterLab:

`uv sync --extra dev` then `uv run jupyter lab` — open `docs/notebooks/temporial_graph_rag_bridge.ipynb`.

## 1. Concept map: reference notebook → this repo

| Reference (`temporal_agents.ipynb`) | This codebase |
|-------------------------------------|----------------|
| `TemporalAgent` (statements, dates, triplets) | `ChunkProcessor` in `temporial_graph_rag.pipeline.processor` |
| OpenAI SDK client | `LLMClient` → HTTP `POST /llm/complete`, `/llm/embeddings`, `GET /llm/models` |
| SQLite + `db_interface` | **Neo4j** via `Neo4jGraphStore` (`temporial_graph_rag.graph.store`) |
| `EntityResolution` + fuzzy DB | Optional `entity_resolution_assist` LLM task + graph `CollectionEntity` / `GlobalEntity` |
| `InvalidationAgent` | Snapshot embedding supersession, event `SUPERSEDES`, extraction `invalidates` / `causes` |
| `factual_qa` / `trend_analysis` / `MultiStepRetriever` | `retrieval.tools`, `retrieval.multi_step.MultiStepRetriever` |
| Earnings HF + `Chunker` | **Out of scope** — clients POST **chunk JSON** (`IngestChunk`) |

See also **[`DEVELOPER_ENHANCEMENTS.md`](../DEVELOPER_ENHANCEMENTS.md)** and **[`DESIGN.md`](../DESIGN.md)**.

## 2. Repository root on `sys.path`

If you opened Jupyter from another folder, adjust `REPO` below.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
if not (REPO / "pyproject.toml").exists():
    # Walk up from docs/notebooks/ to repo root
    for p in [REPO, *REPO.parents]:
        if (p / "pyproject.toml").exists():
            REPO = p
            break
    else:
        raise RuntimeError("Could not find repo root (pyproject.toml). cd to the repo root and restart.")

src = REPO / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

ONTOLOGIES = REPO / "ontologies"
print("REPO =", REPO)

## 3. Load and validate an ontology

Ontology JSON is validated with **[`schemas/ontology.schema.json`](../../schemas/ontology.schema.json)** — see **`docs/ONTOLOGY.md`**.

In [ ]:
from temporial_graph_rag.ontology.loader import load_ontology
from temporial_graph_rag.ontology.schema_validation import validate_ontology_file

path = ONTOLOGIES / "company_events.v1.json"
errs = validate_ontology_file(path)
assert not errs, errs

ontology = load_ontology(ONTOLOGIES, "company_events.v1")
ontology.validate_pair("EARNINGS_FINANCIALS", "RESULTS")
prior = ontology.get_impact_prior("EARNINGS_FINANCIALS", "RESULTS")
print("ontology_id:", ontology.ontology_id)
print("impact prior keys:", sorted(prior.keys()))
print("decay threshold (RESULTS):", ontology.get_decay_weight_threshold("RESULTS"))

## 4. Posted chunk shape (`IngestChunk`)

Matches **IMPLEMENTATION_PLAN** §3: `extraction_text` merges `content` + `title_summary` for text/table; **image** chunks use `title_summary` only.

In [ ]:
from temporial_graph_rag.models.chunk import ChunkType, IngestChunk
from temporial_graph_rag.models.extraction_text import extraction_text_for_ingest

chunk = IngestChunk(
    chunk_id="demo-chunk-1",
    content="Revenue grew 12% YoY.",
    type=ChunkType.TEXT,
    doc_id="filing-2026-Q1",
    bundle_id="bundle-1",
    title_summary="Management discussion",
    publish_date="2026-04-01",
    canonical_event="EARNINGS_FINANCIALS",
    canonical_subevent="RESULTS",
)
print(extraction_text_for_ingest(chunk))

## 5. Event extraction extras (graph alignment)

When persisting to Neo4j, the pipeline can emit:

- **`stable_event_id`** — stable `Event.event_id` for cross-ingest references.
- **`causes_event_ids`** / **`causes`** — **`(:Event)-[:CAUSES]->(:Event)`** edges.
- **`triplets`** in `event_or_triplet_extraction` — **`(:TripletFact)`** + **`ASSERTS_TRIPLET`** from snapshot.

Details: **`DEVELOPER_ENHANCEMENTS.md` §4.3**.

## 6. Optional: ping the FastAPI app

Start the server: `uv run uvicorn temporial_graph_rag.api.main:app --reload --port 8000`

In [ ]:
import os
import urllib.request
import json

base = os.getenv("TEMPORIAL_API_BASE", "http://127.0.0.1:8000").rstrip("/")
try:
    with urllib.request.urlopen(base + "/health", timeout=2) as r:
        print(json.loads(r.read().decode()))
except Exception as e:
    print("API not reachable (start uvicorn):", e)

## 7. Tests

```bash
uv run pytest
```

Neo4j integration tests: **`NEO4J_INTEGRATION_TEST=1`** + credentials — see **`DEVELOPER_ENHANCEMENTS.md` §7.3**.